# Tema 3 — Sistemas modelo

Tema 3 — Sistemas modelo: caja, efecto túnel, oscilador armónico.

Generado por este cuaderno; las figuras se guardan en `../figs/` en PDF vectorial.
Ver `README.md` para el convenio de nombres y los anchos de `tufte-handout`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import eval_hermite
from math import factorial
import qf2figs as qf

## 1. Caja 1D: funciones de onda, densidades y escalera de niveles

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(qf.TEXT, 2.5),
                         gridspec_kw=dict(width_ratios=[1, 1, 0.5]))
L = 1.0
x = np.linspace(0, L, 600)
esc = 0.40
for ax, (func, tit) in zip(
        axes[:2],
        [(lambda n: np.sqrt(2 / L) * np.sin(n * np.pi * x / L), r"$\psi_n(x)$"),
         (lambda n: 2 / L * np.sin(n * np.pi * x / L) ** 2, r"$|\psi_n(x)|^2$")]):
    for n in range(1, 5):
        y = func(n)
        y = y / np.max(np.abs(y)) * esc
        ax.axhline(n, color=qf.MUTED, lw=0.5, ls=":", zorder=0)
        ax.plot(x, y + n, color=qf.PALETTE[n - 1])
        ax.fill_between(x, n, y + n, color=qf.PALETTE[n - 1],
                        alpha=0.16, lw=0)
    ax.axvline(0, color=qf.INK, lw=1.4)
    ax.axvline(L, color=qf.INK, lw=1.4)
    ax.set_xlim(-0.02, L + 0.02)
    ax.set_ylim(0.35, 4.75)
    ax.set_xticks([0, L])
    ax.set_xticklabels(["0", "$L$"])
    ax.set_yticks(range(1, 5))
    ax.set_yticklabels([f"$n={n}$" for n in range(1, 5)], fontsize=7)
    ax.tick_params(axis="y", length=0)
    ax.spines["left"].set_visible(False)
    ax.set_xlabel(r"$x$")
    ax.set_title(tit, loc="left", fontsize=9)
axes[1].set_yticklabels([])
# escalera de niveles a escala real
ax = axes[2]
for n in range(1, 5):
    ax.hlines(n**2, 0, 1, color=qf.PALETTE[n - 1], lw=1.6)
    ax.text(1.08, n**2, f"$n={n}$", fontsize=7, va="center",
            color=qf.PALETTE[n - 1])
ax.annotate("", xy=(0.35, 16), xytext=(0.35, 9),
            arrowprops=dict(arrowstyle="<->", color=qf.MUTED, lw=0.7))
ax.text(0.42, 12.5, r"$\Delta E$", fontsize=7, color=qf.MUTED,
        va="center")
ax.set_xlim(0, 1.9)
ax.set_ylim(0, 18.5)
ax.set_xticks([])
ax.spines["bottom"].set_visible(False)
ax.set_yticks([1, 4, 9, 16])
ax.set_ylabel(r"$E_n \,/\, (h^2/8mL^2)$", fontsize=7)
ax.set_title("niveles", loc="left", fontsize=9)
fig.tight_layout()
qf.save(fig, "caja1d", "psi-psi2", "n1-4")

## 2. Límite clásico

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(qf.TEXT, 1.55), sharey=True)
for ax, n in zip(axes, [1, 5, 30]):
    y = 2 / L * np.sin(n * np.pi * x / L) ** 2
    ax.fill_between(x, 0, y, color=qf.PALETTE[0], alpha=0.30, lw=0)
    ax.plot(x, y, color=qf.PALETTE[0], lw=0.9)
    ax.axhline(1 / L, color=qf.ACCENT, lw=1.2, ls="--")
    ax.set_title(f"$n={n}$", fontsize=8, loc="left")
    ax.set_xticks([0, L])
    ax.set_xticklabels(["0", "$L$"])
    ax.set_xlabel(r"$x$", fontsize=8)
axes[0].set_ylabel(r"$|\psi_n|^2$", fontsize=8)
# etiqueta por encima de las curvas, con trazo hasta la línea clásica
axes[0].set_ylim(0, 2.7)
axes[0].annotate("densidad clásica", xy=(0.12, 1 / L), xytext=(0.02, 2.3),
                 color=qf.ACCENT, fontsize=7,
                 arrowprops=dict(arrowstyle="-", color=qf.ACCENT, lw=0.6))
fig.tight_layout()
qf.save(fig, "caja1d", "limite-clasico")

## 3. Caja 2D: degeneración

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(qf.TEXT, 1.45))
xx, yy = np.meshgrid(np.linspace(0, 1, 200), np.linspace(0, 1, 200))
for ax, (n1, n2) in zip(axes, [(1, 1), (1, 2), (2, 1), (2, 2)]):
    psi = 2 * np.sin(n1 * np.pi * xx) * np.sin(n2 * np.pi * yy)
    ax.imshow(psi, cmap="RdBu_r", vmin=-2, vmax=2, origin="lower",
              extent=(0, 1, 0, 1))
    ax.set_title(rf"$({n1},{n2})$" + "\n" + rf"$E={n1**2 + n2**2}$",
                 fontsize=7.5, loc="left")
    ax.set_xticks([])
    ax.set_yticks([])
for s in axes[1].spines.values():
    s.set(color=qf.ACCENT, lw=1.6, visible=True)
for s in axes[2].spines.values():
    s.set(color=qf.ACCENT, lw=1.6, visible=True)
fig.tight_layout()
qf.save(fig, "caja2d", "degeneracion")

## 4. Efecto túnel

In [ ]:
fig, ax = qf.figure(qf.TEXT, 2.7)
# Solución exacta de la barrera rectangular en unidades con hbar^2/2m = 1/s^2,
# de modo que k = s sqrt(E) y kappa = s sqrt(V - E). Se imponen continuidad
# de psi y de su derivada en x=0 y x=L, con onda incidente A=1 y sin onda
# entrante por la derecha (B'=0). Así la figura cumple las mismas
# condiciones que el texto.
Lb, V0, E, s = 0.5, 1.0, 0.55, 5.0
k = s * np.sqrt(E)
kap = s * np.sqrt(V0 - E)
# incógnitas: B, C, D, A'
M = np.array([
    [-1, 1, 1, 0],
    [1j * k, kap, -kap, 0],
    [0, np.exp(kap * Lb), np.exp(-kap * Lb), -np.exp(1j * k * Lb)],
    [0, kap * np.exp(kap * Lb), -kap * np.exp(-kap * Lb),
     -1j * k * np.exp(1j * k * Lb)],
], dtype=complex)
rhs = np.array([1, 1j * k, 0, 0], dtype=complex)
B, C, D, Ap = np.linalg.solve(M, rhs)
xI = np.linspace(-1.6, 0, 500)
xII = np.linspace(0, Lb, 300)
xIII = np.linspace(Lb, 2.6, 500)
psiI = np.exp(1j * k * xI) + B * np.exp(-1j * k * xI)
psiII = C * np.exp(kap * xII) + D * np.exp(-kap * xII)
psiIII = Ap * np.exp(1j * k * xIII)
# fase global elegida para que psi sea real y máxima en x=0: la parte real
# muestra entonces el decaimiento dentro de la barrera sin cambiar la física
fase = np.exp(-1j * np.angle(psiII[0]))
esc = 0.28 / np.abs(psiII[0])
ax.add_patch(plt.Rectangle((0, 0), Lb, 1.55, color="#ececec", zorder=0))
ax.plot([-1.6, 0, 0, Lb, Lb, 2.6], [0, 0, V0, V0, 0, 0],
        color=qf.INK, lw=1.2)
ax.axhline(E, color=qf.MUTED, lw=0.8, ls="--")
ax.text(Lb / 2, E - 0.05, "$E$", fontsize=8, color=qf.MUTED, ha="center",
        va="top")
ax.text(Lb / 2, V0 + 0.05, "$V$", fontsize=8, color=qf.INK, ha="center")
for xs, ps, col in [(xI, psiI, qf.PALETTE[0]), (xII, psiII, qf.PALETTE[1]),
                    (xIII, psiIII, qf.PALETTE[2])]:
    ax.plot(xs, E + esc * np.real(fase * ps), color=col)
for xc, t, col in [(-0.8, "I: incidente + reflejada", qf.PALETTE[0]),
                   (Lb / 2, "II", qf.PALETTE[1]),
                   (1.55, "III: transmitida", qf.PALETTE[2])]:
    ax.text(xc, 1.35, t, fontsize=7, color=col, ha="center")
ax.set_xlim(-1.6, 2.6)
ax.set_ylim(-0.15, 1.6)
ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.set_xticks([0, Lb])
ax.set_xticklabels(["0", "$L$"])
ax.set_xlabel(r"$x$")
fig.tight_layout()
qf.save(fig, "tunel", "psi-regiones")
# Comprobaciones numéricas (tests de regresión)
print(f"  kappa L      = {kap * Lb:.2f}")
print(f"  R + T        = {abs(B)**2 + abs(Ap)**2:.6f}   (debe ser 1.000000)")
print(f"  T            = {abs(Ap)**2:.3f}")


## 5. Coeficiente de transmisión

In [ ]:
fig, ax = qf.figure(qf.TEXT, 2.4)
hbar, me, eV = 1.054571817e-34, 9.1093837015e-31, 1.602176634e-19
eps = np.linspace(0.001, 0.999, 500)
V = 5 * eV
for i, (nombre, m, Lb) in enumerate([("electrón, $L=0.5$ nm", me, 0.5e-9),
                                     ("electrón, $L=1.0$ nm", me, 1.0e-9),
                                     ("protón,  $L=0.5$ nm", 1836 * me, 0.5e-9)]):
    kap = np.sqrt(2 * m * V * (1 - eps)) / hbar
    T = 16 * eps * (1 - eps) * np.exp(-2 * kap * Lb)
    ax.semilogy(eps, T, color=qf.PALETTE[i])
    qf.label_line(ax, 0.55, T[int(0.55 * len(eps))] * 2.4, nombre,
                  qf.PALETTE[i], fontsize=7.5)
ax.set_ylim(1e-14, 3)
ax.set_xlim(0, 1)
ax.set_xlabel(r"$\varepsilon = E/V$")
ax.set_ylabel(r"$T$  (escala log)")
ax.set_title(r"Transmisión: $T\simeq16\varepsilon(1-\varepsilon)"
             r"\mathrm{e}^{-2\kappa L}$", loc="left")
qf.save(fig, "tunel", "coef-transmision")

## 6. Oscilador armónico

In [ ]:
def psi_ho(v, y):
    N = 1 / np.sqrt(2**v * factorial(v) * np.sqrt(np.pi))
    return N * eval_hermite(v, y) * np.exp(-y**2 / 2)


fig, axes = plt.subplots(1, 2, figsize=(qf.TEXT, 2.9), sharey=True)
y = np.linspace(-5.5, 5.5, 1200)
for ax, (f, tit) in zip(axes, [(lambda v: psi_ho(v, y), r"$\psi_v(y)$"),
                               (lambda v: psi_ho(v, y)**2, r"$|\psi_v(y)|^2$")]):
    ax.plot(y, y**2 / 2, color=qf.INK, lw=1.0)
    for v in range(5):
        Ev = v + 0.5
        f_ = f(v)
        f_ = f_ / np.max(np.abs(f_)) * 0.42
        ax.hlines(Ev, -np.sqrt(2 * Ev), np.sqrt(2 * Ev),
                  color=qf.MUTED, lw=0.5, ls=":")
        ax.plot(y, f_ + Ev, color=qf.PALETTE[v % 6])
        ax.fill_between(y, Ev, f_ + Ev, color=qf.PALETTE[v % 6],
                        alpha=0.16, lw=0)
        # puntos de retorno clásicos
        ax.plot([-np.sqrt(2 * Ev), np.sqrt(2 * Ev)], [Ev, Ev], "|",
                color=qf.ACCENT, ms=6, mew=1.2)
    ax.set_ylim(0, 6.2)
    ax.set_xlim(-5.5, 5.5)
    ax.set_xticks([-4, -2, 0, 2, 4])
    ax.set_yticks([v + 0.5 for v in range(5)])
    ax.set_yticklabels([f"$v={v}$" for v in range(5)], fontsize=7)
    ax.tick_params(axis="y", length=0)
    ax.spines["left"].set_visible(False)
    ax.set_xlabel(r"$y=x/\alpha$")
    ax.set_title(tit, loc="left", fontsize=9)
# con sharey, set_yticklabels([]) borraría también las del panel izquierdo
axes[1].tick_params(labelleft=False)
axes[0].set_ylabel(r"$E_v \,/\, \hbar\omega$")
fig.tight_layout()
qf.save(fig, "osciladorarmonico", "psi-psi2", "v0-4")

## 7. Límite clásico del oscilador

In [ ]:
fig, ax = qf.figure(qf.TEXT, 2.6)
v = 20
Ev = v + 0.5
yt = np.sqrt(2 * Ev)
y = np.linspace(-yt * 1.18, yt * 1.18, 4000)
dens = psi_ho(v, y) ** 2
ax.plot(y, dens, color=qf.PALETTE[0], lw=0.7)
yc = np.linspace(-yt * 0.9995, yt * 0.9995, 2000)
clas = 1 / (np.pi * np.sqrt(2 * Ev - yc**2))
ax.plot(yc, clas, color=qf.ACCENT, lw=1.6, zorder=3)
ax.axvline(-yt, color=qf.MUTED, lw=0.6, ls=":")
ax.axvline(yt, color=qf.MUTED, lw=0.6, ls=":")
# el eje llega por encima del pico cuántico más alto, junto a los puntos
# de retorno; la densidad clásica diverge allí y se corta
ymax = 1.12 * dens.max()
ax.set_ylim(0, ymax)
# etiquetas en la zona libre sobre el centro, con flecha a cada curva
i_q = np.argmax(np.where(np.abs(y + 2.2) < 0.5, dens, 0))
ax.annotate(f"cuántica, $v={v}$", xy=(y[i_q], dens[i_q]),
            xytext=(-3.2, 0.19), color=qf.PALETTE[0], fontsize=7.5,
            ha="center", arrowprops=dict(arrowstyle="-", color=qf.PALETTE[0],
                                         lw=0.6))
yl = 3.0
ax.annotate(r"clásica $\propto 1/\sqrt{E-V}$",
            xy=(yl, 1 / (np.pi * np.sqrt(2 * Ev - yl**2))),
            xytext=(3.2, 0.19), color=qf.ACCENT, fontsize=7.5, ha="center",
            arrowprops=dict(arrowstyle="-", color=qf.ACCENT, lw=0.6))
ax.set_xlabel(r"$y=x/\alpha$")
ax.set_ylabel("densidad de probabilidad")
qf.save(fig, "osciladorarmonico", "limite-clasico", "v20")